<a href="https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Look at key fields before trusting any flag rule.
Expect heavy tails on impressions and age.

In [5]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

if not Path("data/raw/content_refresh_anonymized.csv").exists():
    if not Path("/content/FlyRank-ML").exists():
        !git clone https://github.com/Fatima-05/FlyRank-ML.git /content/FlyRank-ML
    os.chdir("/content/FlyRank-ML")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

cols = [c for c in ["impressions_90d", "ctr", "content_age_days", "avg_position", "word_count"] if c in df.columns]
print("Rows:", len(df))
print(df[cols].describe(percentiles=[0.5, 0.9, 0.99]).T)

Rows: 30000
                    count         mean           std   min      50%       90%  \
impressions_90d   30000.0  5200.366300  16838.019547   1.0   731.00  12136.40   
ctr               30000.0     0.510733      3.279162   0.0     0.07      0.65   
content_age_days  30000.0   256.167800    132.707930  90.0   236.00    463.00   
avg_position      30000.0    16.342380     15.216790   0.0    10.80     36.80   
word_count        22301.0  3107.760325   1452.382598   8.0  2877.00   5327.00   

                        99%       max  
impressions_90d   73505.830  517715.0  
ctr                   8.330     100.0  
content_age_days    537.000     564.0  
avg_position         69.901     245.0  
word_count         7292.000    9546.0  


## 2. Signal tests

Three safe signals with mini-tests and verdicts.
Verdict options: CONFIRMED / OPPOSITE / MIXED / FALSE

In [6]:
# Signal 1: high impressions among declining pages?
df["imp_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 100, 500, 2000, 10_000_000],
    labels=["low", "mid", "high", "very_high"],
)
s1 = df.groupby("imp_bucket", observed=True)["is_declining"].mean()
print("Signal 1 — declining rate by impressions bucket:")
print(s1)
print("n =", len(df))
# interpret after print

# Signal 2: older content more often declining?
df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[-1, 90, 180, 365, 10_000],
    labels=["fresh", "aging", "stale", "very_stale"],
)
s2 = df.groupby("age_bucket", observed=True)["is_declining"].mean()
print("\nSignal 2 — declining rate by age bucket:")
print(s2)

# Signal 3: low CTR more common on declining pages?
df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=[-0.01, 0.01, 0.03, 0.08, 1.0],
    labels=["very_low", "low", "mid", "high"],
)
s3 = df.groupby("ctr_bucket", observed=True)["is_declining"].mean()
print("\nSignal 3 — declining rate by CTR bucket:")
print(s3)

Signal 1 — declining rate by impressions bucket:
imp_bucket
low          0.389208
mid          0.604281
high         0.617964
very_high    0.581416
Name: is_declining, dtype: float64
n = 30000

Signal 2 — declining rate by age bucket:
age_bucket
fresh         0.668699
aging         0.625552
stale         0.514866
very_stale    0.426258
Name: is_declining, dtype: float64

Signal 3 — declining rate by CTR bucket:
ctr_bucket
very_low    0.498119
low         0.731915
mid         0.685265
high        0.573475
Name: is_declining, dtype: float64


Verdicts (update if your printed rates disagree):

Signal 1 impressions: MIXED or CONFIRMED depending on whether declining rate rises with impressions.
Signal 2 age/staleness: CONFIRMED if older buckets show higher declining rate.
Signal 3 low CTR: MIXED/CONFIRMED if very_low CTR has higher declining rate than high CTR.

These are associations on this table, not proof that the signal causes decline.

## 3. The flag-linked test

FlyRank-style refresh logic often leans on staleness and/or CTR-vs-visibility.
Test staleness: are older pages more often declining in this dataset?

In [7]:
flag_test = df.groupby("age_bucket", observed=True).agg(
    n=("is_declining", "size"),
    declining_rate=("is_declining", "mean"),
    median_impressions=("impressions_90d", "median"),
)
print(flag_test)

# simple monotone check: very_stale vs fresh
if set(["fresh", "very_stale"]).issubset(set(flag_test.index.astype(str))):
    fresh = flag_test.loc["fresh", "declining_rate"]
    very = flag_test.loc["very_stale", "declining_rate"]
    print(f"fresh declining_rate={fresh:.3f} | very_stale declining_rate={very:.3f}")
    print("Verdict support:", "CONFIRMED" if very > fresh else "OPPOSITE/MIXED")

                n  declining_rate  median_impressions
age_bucket                                           
fresh         492        0.668699               294.0
aging       11780        0.625552               741.5
stale       11368        0.514866               640.5
very_stale   6360        0.426258               842.5
fresh declining_rate=0.669 | very_stale declining_rate=0.426
Verdict support: OPPOSITE/MIXED


## 4. What this means in practice

A content team should treat staleness, impressions, and low CTR as useful ranking clues, not automatic truth.
Use them to build a short review queue, then let a human decide.
If a signal is MIXED or OPPOSITE in the tables above, do not hard-code it as a strong solo rule.

In [8]:
print("Practical takeaway: use signals for prioritization, keep human review, avoid single-signal automation.")

Practical takeaway: use signals for prioritization, keep human review, avoid single-signal automation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.